# 05f - HayFlow-Hines segment-conditioned neural micro-canary

This notebook tests whether the rank-64/96 capacity observed in 05e generalizes beyond the fitted pair. It trains only zero-output segment-conditioned residual heads on multiple `train` counterfactual pairs and evaluates on disjoint test-split pairs. The H2 base and its feature extractor remain frozen. No rollout or full training is present.

## 1. Coherent checkout and GPU runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml', 'matplotlib'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05f.'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0), 'experiment': 'fresh held-out neural micro-canary'})

## 2. Inputs
Sono richiesti il composite targeted e gli artefatti esatti 05b, 05c, 05d e 05e. ZIP originali e directory estratte da Kaggle sono entrambi accettati e verificati tramite SHA-256.

In [ ]:
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'
    stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp)
    return destination

topup_override = os.environ.get('HAYFLOW_TOPUP_V3')
topup_candidates = [Path(topup_override).expanduser()] if topup_override else []
topup_candidates.extend(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates.extend(path.parent for path in INPUT_ROOT.rglob('composite_dataset_manifest.json'))
TOPUP_SOURCE = next((p.resolve() for p in topup_candidates if p.exists()), None)
assert TOPUP_SOURCE is not None, 'Top-up BAP v3 non trovato.'
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05f_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]

base_override = os.environ.get('HAYFLOW_BASE_DATASET')
base_candidates = [Path(base_override).expanduser()] if base_override else []
base_candidates.extend(path.parent for path in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(path).lower() and 'topup' not in str(path).lower())
base_candidates.extend(path for path in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(path).lower())
BASE_SOURCE = next((p.resolve() for p in base_candidates if p.exists()), None)
assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'

checkpoint_override = os.environ.get('HAYFLOW_05B_ARTIFACT')
checkpoint_candidates = [Path(checkpoint_override).expanduser()] if checkpoint_override else []
checkpoint_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_canary_v2.zip'))
checkpoint_candidates.extend(path.parent.parent for path in INPUT_ROOT.rglob('canary_models.pt') if path.parent.name == 'checkpoints' and 'hayflow' in str(path).lower())
CHECKPOINT_05B_SOURCE = next((p.resolve() for p in checkpoint_candidates if p.exists()), None)
assert CHECKPOINT_05B_SOURCE is not None, 'Artefatto 05b non trovato.'

causal_override = os.environ.get('HAYFLOW_05C_ARTIFACT')
causal_candidates = [Path(causal_override).expanduser()] if causal_override else []
causal_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_causal_isolation.zip'))
causal_candidates.extend(path.parent for path in INPUT_ROOT.rglob('final_report.json') if (path.parent / 'checkpoint_forensics.json').is_file() and (path.parent / 'progressive_isolation_report.json').is_file())
ARTIFACT_05C_SOURCE = next((p.resolve() for p in causal_candidates if p.exists()), None)
assert ARTIFACT_05C_SOURCE is not None, 'Artefatto 05c non trovato.'

conditioning_override = os.environ.get('HAYFLOW_05D_ARTIFACT')
conditioning_candidates = [Path(conditioning_override).expanduser()] if conditioning_override else []
conditioning_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_residual_conditioning.zip'))
conditioning_candidates.extend(path.parent for path in INPUT_ROOT.rglob('final_report.json') if (path.parent / 'free_residual_report.json').is_file() and (path.parent / 'unfreezing_ladder_report.json').is_file())
ARTIFACT_05D_SOURCE = next((p.resolve() for p in conditioning_candidates if p.exists()), None)
assert ARTIFACT_05D_SOURCE is not None, 'Artefatto 05d non trovato.'

capacity_override = os.environ.get('HAYFLOW_05E_ARTIFACT')
capacity_candidates = [Path(capacity_override).expanduser()] if capacity_override else []
capacity_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_segment_capacity.zip'))
capacity_candidates.extend(path.parent for path in INPUT_ROOT.rglob('final_report.json') if (path.parent / 'capacity_probe_report.json').is_file() and (path.parent / 'capacity_probe_metrics.parquet').is_file())
ARTIFACT_05E_SOURCE = next((p.resolve() for p in capacity_candidates if p.exists()), None)
assert ARTIFACT_05E_SOURCE is not None, 'Artefatto 05e hayflow_hines_segment_capacity non trovato.'
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), '05b': str(CHECKPOINT_05B_SOURCE), '05c': str(ARTIFACT_05C_SOURCE), '05d': str(ARTIFACT_05D_SOURCE), '05e': str(ARTIFACT_05E_SOURCE)})

## 3. Composite and cryptographic provenance preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now)
    percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9)
        eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05f][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True)
        hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
bundle_summary = {'valid': bool(bundle.manifest['valid']), 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count, 'physical_merge_performed': bool(bundle.manifest['physical_merge_performed'])}
display(bundle_summary)
assert bundle_summary['valid'] and bundle_summary['transition_count'] == 29880
assert not bundle_summary['physical_merge_performed']

In [ ]:
from src.hayflow_model import HinesCapacityConfig, HinesConditioningConfig, HinesIsolationConfig, HinesPrototypeExperimentConfig, HinesSegmentCanaryConfig, HinesSegmentMicroCanaryExperiment
raw = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_segment_micro_canary.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(raw['model_experiment'])
isolation_config = HinesIsolationConfig.from_mapping(raw['isolation'])
conditioning_config = HinesConditioningConfig.from_mapping(raw['conditioning'])
capacity_config = HinesCapacityConfig.from_mapping(raw['capacity'])
canary_config = HinesSegmentCanaryConfig.from_mapping(raw['micro_canary'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_segment_micro_canary')
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
session = HinesSegmentMicroCanaryExperiment(bundle, OUTPUT_DIR, model_config, isolation_config, conditioning_config, capacity_config, canary_config, CHECKPOINT_05B_SOURCE, ARTIFACT_05C_SOURCE, ARTIFACT_05D_SOURCE, ARTIFACT_05E_SOURCE, code_revision=REVISION)
prepare_report = session.prepare_micro_canary()
display(prepare_report)
assert prepare_report['training_contract_blockers'] == []
assert not prepare_report['full_training_authorized']

## 4. Leakage-safe counterfactual pair plan
La coppia 05e è esclusa dal training. Le coppie di ottimizzazione provengono solo da `train`; quelle held-out solo dagli split controfattuali di test. Ogni coppia ha stato completo iniziale identico entro tolleranza, input `U_realized` differente ed episodi disgiunti.

In [ ]:
pair_plan = session.build_pair_plan()
display({
    'valid': pair_plan['valid'],
    'train_pairs': pair_plan['train_pair_count'],
    'heldout_pairs': pair_plan['heldout_pair_count'],
    'heldout_splits': pair_plan['heldout_splits'],
    'heldout_candidate_counts': pair_plan['heldout_candidate_counts'],
    'development_pair_excluded': pair_plan['development_pair_excluded_from_training'],
    'episode_overlap': pair_plan['episode_overlap'],
    'pair_plan_sha256': pair_plan['pair_plan_sha256'],
})
assert pair_plan['valid']
assert pair_plan['train_pair_count'] >= canary_config.minimum_train_pair_count
assert pair_plan['heldout_pair_count'] >= canary_config.minimum_heldout_pair_count
assert pair_plan['development_pair_excluded_from_training']
assert pair_plan['episode_overlap'] == []

## 5. Rank-64 and rank-96 neural micro-canaries
Ogni residuo parte esattamente da zero. La base H2 è congelata. La normalizzazione e la base spettrale vengono calcolate solo sulle coppie train. I checkpoint sono scelti esclusivamente dalla loss train; i target held-out vengono letti soltanto per la valutazione finale. Durante questa cella vengono stampati avanzamento ed ETA.

In [ ]:
micro_report = session.run_micro_canary()
display({
    'valid': micro_report['valid'],
    'smallest_passing_heldout_rank': micro_report['smallest_passing_heldout_rank'],
    'runs': [
        {
            'rank': run['rank'],
            'zero_initialized': run['zero_initialized'],
            'train_passed': run['passed_train'],
            'heldout_passed': run['passed_heldout'],
            'train_rmse_mv': run['evaluations']['train']['aggregate_voltage_rmse_mv'],
            'heldout_rmse_mv': run['evaluations']['heldout']['aggregate_voltage_rmse_mv'],
            'heldout_max_error_mv': run['evaluations']['heldout']['maximum_segment_error_mv'],
            'heldout_retention_median': run['evaluations']['heldout']['median_branching_retention'],
        } for run in micro_report['runs']
    ],
})
assert micro_report['valid']
assert not micro_report['full_training_authorized']

## 6. Inspect train versus genuinely held-out behavior

In [ ]:
metrics = pd.read_parquet(OUTPUT_DIR / 'micro_canary_metrics.parquet')
display(metrics)
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / 'micro_canary_diagnostics.png')))

## 7. Diagnostic verdict and artifact contract

In [ ]:
final_report = session.finalize_micro_canary(micro_report)
display(final_report)
assert final_report['valid']
assert not final_report['full_training_authorized']
required = ['micro_canary_config.json', 'pair_plan.json', 'pair_plan.parquet', 'feature_contract.json', 'spectral_basis_report.json', 'micro_canary_report.json', 'micro_canary_training_history.parquet', 'micro_canary_metrics.parquet', 'micro_canary_diagnostics.png', 'rank_64_report.json', 'rank_96_report.json', 'final_report.json', 'artifact_index.json', 'checkpoints/rank_64.pt', 'checkpoints/rank_96.pt']
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
assert not missing, missing
print({'diagnosis': final_report['diagnosis'], 'selected_rank': final_report['selected_rank_for_next_diagnostic'], 'next': final_report['next_step'], 'output_dir': str(OUTPUT_DIR)})

## 8. Browser download with checkpoints

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_segment_micro_canary')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f'''
const binary = atob('{encoded}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
'''))
print('Download avviato:', filename, f'({zip_path.stat().st_size / 2**20:.1f} MiB)')